# Bloc 7 — Compléments : Merge, Export & Visualisation Express
Objectifs: fusionner des DataFrames (jointures), exporter les résultats, visualiser rapidement

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Fusionner des DataFrames — Les Jointures
Comme en SQL, on fusionne des tables grâce à une clé commune. Pandas propose `pd.merge()` avec 4 types de jointures.

### Les 4 types de jointures
```
inner : garde uniquement les clés communes (intersection)
left  : garde toutes les clés de la table de gauche
right : garde toutes les clés de la table de droite
outer : garde toutes les clés des deux tables (union)
```

### Exemple simple

In [2]:
clients = pd.DataFrame({'id': [1, 2, 3], 'nom': ['Alice', 'Bob', 'Charlie']})
commandes = pd.DataFrame({'client_id': [1, 1, 4], 'montant': [100, 50, 200]})

# Observez : seuls les clients présents dans les deux tables apparaissent
inner_join = pd.merge(clients, commandes, left_on='id', right_on='client_id', how='inner')
display(inner_join)

,id,nom,client_id,montant
0,1,Alice,1,100
1,1,Alice,1,50


### Left join : garder tout le monde à gauche

In [3]:
# Le client sans commande apparait avec NaN
left_join = pd.merge(clients, commandes, left_on='id', right_on='client_id', how='left')
display(left_join)

,id,nom,client_id,montant
0,1,Alice,1.0,100.0
1,1,Alice,1.0,50.0
2,2,Bob,NaN,NaN
3,3,Charlie,NaN,NaN


### Mini-exercice : testez les autres jointures

In [4]:
# Testez un outer join — que se passe-t-il ?
resultat = pd.merge(clients, commandes, left_on='id', right_on='client_id', how='___')
display(resultat)
# Combien de lignes ? Pourquoi ?

ValueError: '___' is not a valid Merge type: left, right, inner, outer, left_anti, right_anti, cross, asof

### Démo live : reconstruire les ventes TheLook
Le fichier `thelook_sales.csv` est en fait le résultat de plusieurs jointures. Reconstruisons-le.

In [5]:
order_items = pd.read_csv('../01_data/raw/thelook_order_items.csv')
products = pd.read_csv('../01_data/raw/thelook_products.csv')

# Étape 1 : joindre les articles aux produits
ventes = pd.merge(order_items, products, left_on='product_id', right_on='id', suffixes=('', '_product'))
print(f"Après jointure items × products : {ventes.shape}")
display(ventes.head(3))

Après jointure items × products : (50000, 19)


,id,order_id,user_id,product_id,sale_price,status,created_at,shipped_at,delivered_at,returned_at,id_product,name,category,brand,department,cost,retail_price,sku,distribution_center_id
0,1,2006,3641,533,37.47,Complete,2023-05-24 05:04:44,2023-05-25 05:04:44,2023-05-30 05:04:44,NaN,533,Volcom Pro Active,Active,Volcom,Women,27.45,50.62,SKU-00533,5
1,2,19606,4289,1234,41.01,Complete,2024-06-14 02:09:32,2024-06-16 02:09:32,2024-06-19 02:09:32,NaN,1234,Volcom Modern Shorts,Shorts,Volcom,Men,23.79,43.15,SKU-01234,6
2,3,14234,973,409,15.67,Complete,2023-06-16 03:42:36,2023-06-17 03:42:36,2023-06-25 03:42:36,NaN,409,Adidas Modern Socks & Hosiery,Socks & Hosiery,Adidas,Men,8.97,15.95,SKU-00409,8


In [6]:
users = pd.read_csv('../01_data/raw/thelook_users.csv')

# Étape 2 : joindre les utilisateurs
ventes = pd.merge(ventes, users, left_on='user_id', right_on='id', suffixes=('', '_user'))
print(f"Après jointure + users : {ventes.shape}")
# C'est exactement ce qu'est thelook_sales.csv !

Après jointure + users : (50000, 29)


## 2. Empiler des DataFrames : pd.concat()

### Empiler verticalement (ajouter des lignes)

In [ ]:
df1 = pd.DataFrame({'A': [1, 2]})
df2 = pd.DataFrame({'A': [3, 4]})

# Utile quand on a des données mensuelles dans des fichiers séparés
empile = pd.concat([df1, df2], ignore_index=True)
display(empile)

## 3. Exporter ses résultats

### Sauvegarder en CSV

In [ ]:
top_categories = ventes.groupby('category')['sale_price'].sum().reset_index()
os.makedirs('../03_analysis/reports/', exist_ok=True)

# Toujours mettre index=False sauf si l'index a un sens métier
top_categories.to_csv('../03_analysis/reports/top_categories.csv', index=False)

### Sauvegarder en Pickle (`.to_pickle`)

Le format **Pickle (`.pkl`)** est le format binaire natif de Python pour sérialiser et sauvegarder des objets directement sur le disque.

#### 🚀 Pourquoi utiliser Pickle plutôt que CSV ?
1. **Conservation parfaite des types (`dtypes`) :** 
   - En CSV, tout redevient du texte brut (les dates redeviennent des chaînes de caractères `str`, les catégories sont perdues).
   - En Pickle, **les types exacts sont conservés** (`datetime64`, `category`, `int32`, `float64`, etc.). Pas besoin de refaire `pd.to_datetime()` au rechargement !
2. **Vitesse :** L'écriture et la lecture sont **nettement plus rapides** que le CSV car il n'y a aucun parsing textuel.
3. **Préserve l'index :** L'index (même hiérarchique / MultiIndex) est rechargé exactement à l'identique.

> ⚠️ **Attention aux limites :** 
> - **Python uniquement :** Un fichier `.pkl` ne peut pas être ouvert dans Excel ou lu facilement par d'autres langages (R, Java, etc.). Pour un format binaire universel et rapide en Big Data, on préférera **Parquet** (`.parquet`).
> - **Sécurité :** Ne chargez jamais un fichier pickle provenant d'une source inconnue.


In [ ]:
# 1. Sauvegarde au format Pickle (.pkl)
chemin_pickle = '../03_analysis/reports/top_categories.pkl'
top_categories.to_pickle(chemin_pickle)
print(f"Fichier sauvegardé avec succès dans : {chemin_pickle}")

# 2. Rechargement instantané avec read_pickle()
df_restaure = pd.read_pickle(chemin_pickle)

# 3. Vérification : tous les types d'origine sont intacts !
print("\nTypes de données restaurés à l'identique :")
print(df_restaure.dtypes)

## 4. Visualisation Express avec .plot()
Ce n'est PAS un cours de dataviz. Mais `.plot()` est indispensable pour vérifier rapidement ses résultats.

### Barplot horizontal

In [ ]:
top_countries = ventes.groupby('country')['sale_price'].sum().nlargest(10).sort_values()
top_countries.plot(kind='barh', title='Top 10 Pays par CA')
plt.show()

### Line plot

In [ ]:
ventes['created_at'] = pd.to_datetime(ventes['created_at'])
sales_by_month = ventes.set_index('created_at').resample('ME')['sale_price'].sum()

# Idéal pour voir les tendances temporelles
sales_by_month.plot(kind='line', title='Ventes Mensuelles')
plt.show()

### Histogramme

In [ ]:
# Quelle est la distribution des prix ?
ventes['sale_price'].plot(kind='hist', bins=50, title='Distribution des prix')
plt.show()

### À vous : scatter plot

In [ ]:
# Tracez un nuage de points : cost en X, sale_price en Y
ventes.plot(kind='scatter', x='___', y='___', alpha=0.1)
plt.title('___')
plt.show()

---
# Exercices pratiques

## Exercice 7.1 — La Jointure
1. Chargez `thelook_orders.csv` et `thelook_users.csv`
2. Fusionnez-les avec un left join sur user_id / id
3. Combien de lignes obtenez-vous ? Est-ce le même nombre que orders ? Pourquoi ?

In [ ]:
# 1. Chargement
# Votre code ici

In [ ]:
# 2 et 3. Merge + analyse
# Votre code ici

## Exercice 7.2 — Export & Visualisation
1. À partir du merge précédent, calculez le CA total par pays
2. Prenez le top 10 et tracez un barplot horizontal
3. Exportez le résultat dans un autre format que.csv dans `../03_analysis/reports/`

In [ ]:
# 1 et 2. Calcul + visualisation
# Votre code ici

In [ ]:
# 3. Export (meilleur format)
# Votre code ici

---
## Bonus 

Refaites l'exercice 4 avec la library Seaborn


In [ ]:
! uv add seaborn

In [ ]:
# votre code ici